# Solar Filament Segmentation 2026 - final pipeline (inference only)

This notebook reproduces our **submitted predictions without any training**:

1. verify the trained weights committed to this repository (weights/best_det.pt, weights/best_ref.pt),
2. run the frozen detector -> crop-refiner cascade over the 180 official test images,
3. assemble pixel-disjoint instances (the official scorer rejects overlapping masks),
4. encode compressed COCO RLE and write submission.csv,
5. validate the file (schema, coverage, RLE decoding, per-image overlap, positive area).

Expected runtime: about 2 minutes for all 180 test images on one NVIDIA T4 (CPU works too, slower).
The inference code is imported from src/cloud_pipeline.py - the same module that produced the
submitted CSV - so no numerical logic is duplicated here.


In [ ]:
import os
import sys
from pathlib import Path

import torch

REPO = Path.cwd().resolve()
while not (REPO / "requirements.txt").exists() and REPO.parent != REPO:
    REPO = REPO.parent
assert (REPO / "requirements.txt").exists(), "run this notebook from inside the cloned repository"
sys.path.insert(0, str(REPO))

print("python :", sys.version.split()[0])
print("torch  :", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu    :", torch.cuda.get_device_name(0))
print("repo   :", REPO)


In [ ]:
import yaml

CFG = yaml.safe_load(open(REPO / "configs/inference.yaml", encoding="utf-8"))
data_override = os.environ.get("FILAMENT_DATA_ROOT")
DATA_ROOT = Path(data_override) if data_override else REPO / CFG["data_root"]
TEST_DIR = DATA_ROOT / CFG["test_dir"]
test_images = sorted(TEST_DIR.glob("*.jpeg"))
smoke_limit = int(os.environ.get("SMOKE_LIMIT", "0"))
if smoke_limit:
    test_images = test_images[:smoke_limit]
print("data root   :", DATA_ROOT)
print("test images :", len(test_images))
assert len(test_images) >= 1, (
    "official test images not found - download the competition data and unzip it so that "
    "data/MAGFiLO_1.0_Kaggle_2026/test/test_images exists (see docs/INFERENCE.md)"
)
device = "cuda:0" if torch.cuda.is_available() else "cpu"


In [ ]:
import hashlib

# hashes frozen in weights/README.md
EXPECTED = {
    "best_det.pt": (
        "fd91ebd028e0ab58d7ff726013104c92",
        "fd3f534f770b7027ba900bd4d96b3559e5888b612868ec86eaf3ae16b8396814",
    ),
    "best_ref.pt": (
        "678a319cb4bd58d507ea89ea3b1d3a7f",
        "581280342f1ee04ea0d0cef6f8fceb449264392e5c9fe737e03f74894e618409",
    ),
}


def hash_file(path, algorithm):
    h = hashlib.new(algorithm)
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


for name, (md5_expected, sha256_expected) in EXPECTED.items():
    path = REPO / "weights" / name
    assert path.is_file(), f"missing weights/{name} (see weights/README.md)"
    md5_got = hash_file(path, "md5")
    sha256_got = hash_file(path, "sha256")
    assert md5_got == md5_expected and sha256_got == sha256_expected, f"hash mismatch for {name}"
    print(f"weights/{name}: {path.stat().st_size / 1e6:.1f} MB  md5={md5_got}  OK")


In [ ]:
from argparse import Namespace

from src.cloud_pipeline import (
    build_refiner_class,
    entries_for_config,
    entry_to_full,
    load_detector,
    predict_image,
)

args = Namespace(
    img_size=int(CFG["detector"]["imgsz"]),
    ref_crop=int(CFG["refiner"]["size"]),
)
detector = load_detector(REPO / CFG["weights"]["detector"], args)

SeedRefiner = build_refiner_class()
payload = torch.load(REPO / CFG["weights"]["refiner"], map_location="cpu", weights_only=False)
state = payload["model"] if isinstance(payload, dict) and "model" in payload else payload
refiner = SeedRefiner(pretrained=False)
refiner.load_state_dict(state)
refiner = refiner.to(device).eval()
print("models loaded from", CFG["weights"])


In [ ]:
import time

import numpy as np
import pandas as pd
from pycocotools import mask as mask_utils
from tqdm.auto import tqdm

H = W = int(CFG["output"]["image_size"])
BEST = {
    "conf": float(CFG["detector"]["conf"]),
    "ref": float(CFG["refiner"]["threshold"]),
    "tta": str(CFG["refiner"]["tta"]),
}
ref_crop = int(CFG["refiner"]["size"])
imgsz = int(CFG["detector"]["imgsz"])

rows = []
overlap_errors = 0
started = time.time()
for path in tqdm(test_images, desc="test images"):
    refined = predict_image(
        path, detector, refiner, device, BEST["conf"], args, imgsz=imgsz, ref_crop=ref_crop
    )
    entries = entries_for_config(refined, BEST["conf"], BEST["ref"], BEST["tta"])
    occupied = np.zeros((H, W), dtype=bool)
    for index, entry in enumerate(entries, 1):
        full = entry_to_full(entry)
        if np.any(np.logical_and(full, occupied)):
            overlap_errors += 1
        occupied |= full
        counts = mask_utils.encode(np.asfortranarray(full.astype(np.uint8)))["counts"]
        rows.append(
            {
                "filament_id": f"{path.stem}_{index}",
                "segmentation_rle": counts.decode("ascii") if isinstance(counts, bytes) else str(counts),
            }
        )
elapsed = time.time() - started
print(f"inference: {len(test_images)} images in {elapsed / 60:.1f} min, {len(rows)} instances, "
      f"overlap_errors={overlap_errors}")
assert overlap_errors == 0, "overlapping predictions would be rejected by the official scorer"

submission = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
out_path = REPO / CFG["output"]["submission_path"]
submission.to_csv(out_path, index=False)
print("wrote", out_path, submission.shape[0], "rows")


In [ ]:
import re

H = W = int(CFG["output"]["image_size"])
test_stems = {path.stem for path in sorted(TEST_DIR.glob("*.jpeg"))}


def validate(path, expected_stems):
    df = pd.read_csv(path, dtype=str)
    errors = []
    if list(df.columns) != ["filament_id", "segmentation_rle"]:
        errors.append(f"columns={list(df.columns)}")
    ids = df["filament_id"]
    if not ids.is_unique:
        errors.append("duplicate filament_id")
    if not ids.str.match(r"^[A-Za-z0-9]+_\d+$").all():
        errors.append("malformed filament_id")
    stems = ids.str.rsplit("_", n=1).str[0]
    unknown = sorted(set(stems) - expected_stems)
    if unknown:
        errors.append(f"unknown image ids {unknown[:3]}")
    bad_decode = empty = 0
    areas = []
    masks_by_image = {}
    for fid, rle in zip(ids, df["segmentation_rle"]):
        try:
            mask = mask_utils.decode({"size": [H, W], "counts": rle.encode("ascii")})
        except Exception:
            bad_decode += 1
            continue
        if mask.shape != (H, W):
            bad_decode += 1
            continue
        area = int(mask.sum())
        if area == 0:
            empty += 1
        areas.append(area)
        masks_by_image.setdefault(fid.rsplit("_", 1)[0], []).append(mask)
    if bad_decode:
        errors.append(f"undecodable RLE x{bad_decode}")
    if empty:
        errors.append(f"empty masks x{empty}")
    overlap_images = 0
    for stem, masks in masks_by_image.items():
        acc = np.zeros((H, W), dtype=np.uint16)
        for mask in masks:
            acc += mask.astype(np.uint16)
        overlap_images += int((acc > 1).any())
    if overlap_images:
        errors.append(f"overlapping masks in {overlap_images} images")
    print(f"rows={len(df)}  images={len(masks_by_image)}/{len(expected_stems)}  "
          f"area[min/med/max]={min(areas)}/{int(np.median(areas))}/{max(areas)}")
    print("VALIDATION:", "PASS" if not errors else f"FAILED {errors}")
    assert not errors, errors
    return df


validated = validate(out_path, test_stems)
print("instances per image (mean):", round(len(validated) / max(len(masks_by_image), 1), 2))


## Expected result

Running the full 180-image test set with the committed weights reproduces the fold-0 submission of
our YOLO11-seg + crop-refiner cascade:

| quantity | value |
|---|---|
| instances | 1065 rows |
| images with predictions | 177 / 180 |
| fold-0 validation PQ (official pooled replica) | 0.3780 (SQ 0.6475, RQ 0.5838) |
| public leaderboard | 0.34 |

The better public score reported by the team (0.39, ref 57768486) comes from a public U-Net
ensemble reproduced from the Kaggle dataset dariushafshar/solar-filament-unet-weights; see the
top-level README and the technical report (docs/report.pdf) for that submission.
